# LoRA vs Full Fine-Tuning on RoBERTa-base

This notebook clones the project repo, installs dependencies, and runs all six
experiments:

| Mode | Dataset | Description |
|---|---|---|
| baseline | SST-2 | Zero-shot evaluation (no training) |
| finetune | SST-2 | Full fine-tuning |
| lora | SST-2 | LoRA fine-tuning |
| baseline | MNLI | Zero-shot evaluation (no training) |
| finetune | MNLI | Full fine-tuning |
| lora | MNLI | LoRA fine-tuning |

Results are collected and printed as a comparison table at the end.

## 1. Setup

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
REPO_URL      = "https://github.com/da-luce/cs5782_final_project.git"
BRANCH        = "baseline"   # must be "main" for the final submission
TRAIN_SAMPLES = 50000
VAL_SAMPLES   = 5000
EPOCHS        = 3
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
REPO_DIR = REPO_URL.split("/")[-1].replace(".git", "")

!git clone -b {BRANCH} {REPO_URL}
%cd {REPO_DIR}

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. SST-2 Experiments

#### SST-2 Dataset

SST-2 is a sentiment analysis dataset, and a component of the GLUE benchmark. Each sentence is manually annotated with a binary label (positive or negative) to indicate the sentiment expressed by the reviewer.

This is a relatively **simple** task.

In [ ]:
!python code/train.py \
    --mode baseline \
    --dataset sst2 \
    --val_samples {VAL_SAMPLES}

In [ ]:
!python code/train.py \
    --mode finetune \
    --dataset sst2 \
    --train_samples {TRAIN_SAMPLES} \
    --val_samples {VAL_SAMPLES} \
    --epochs {EPOCHS}

In [ ]:
!python code/train.py \
    --mode lora \
    --dataset sst2 \
    --train_samples {TRAIN_SAMPLES} \
    --val_samples {VAL_SAMPLES} \
    --epochs {EPOCHS}

## 3. MNLI Experiments

#### MNLI Dataset

MNLI (Multi-Genre Natural Language Inference) is a large dataset and a key component of the GLUE benchmark. Each entry consists of a sentence pair--a premise and a hypothesis--manually annotated with a label of entailment, contradiction, or neutral.

MNLI includes a diverse range of written and spoken English across ten different genres. This makes it a significantly more **complex** task that tests a model's ability to perform logical reasoning and generalization.

In [ ]:
!python code/train.py \
    --mode baseline \
    --dataset mnli \
    --val_samples {VAL_SAMPLES}

In [ ]:
!python code/train.py \
    --mode finetune \
    --dataset mnli \
    --train_samples {TRAIN_SAMPLES} \
    --val_samples {VAL_SAMPLES} \
    --epochs {EPOCHS}

In [ ]:
!python code/train.py \
    --mode lora \
    --dataset mnli \
    --train_samples {TRAIN_SAMPLES} \
    --val_samples {VAL_SAMPLES} \
    --epochs {EPOCHS}

## 4. Results Comparison

In [ ]:
import json
import os

experiments = [
    ("baseline", "sst2"),
    ("finetune", "sst2"),
    ("lora",     "sst2"),
    ("baseline", "mnli"),
    ("finetune", "mnli"),
    ("lora",     "mnli"),
]

rows = []
for mode, dataset in experiments:
    path = f"results/{mode}_{dataset}.json"
    if not os.path.exists(path):
        print(f"Missing results for {mode}/{dataset} — skipping")
        continue

    with open(path) as f:
        info = json.load(f)

    accuracy = info["eval"]["results"].get("eval_accuracy", float("nan"))
    trainable = info["model"]["trainable"]
    trainable_pct = info["model"]["trainable_pct"]
    total = info["model"]["total"]

    # Calculate total time from the segmented timers
    train_time = info["training"]["time_sec"]
    eval_time = info["eval"]["time_sec"]
    time_min = (train_time + eval_time) / 60

    rows.append((mode, dataset, accuracy, trainable, trainable_pct, total, time_min))

header = f"{'Mode':<12} {'Dataset':<8} {'Accuracy':>10} {'Trainable params':>18} {'Trainable %':>12} {'Total params':>14} {'Time (min)':>12}"
sep = "-" * len(header)
print(header)
print(sep)
for mode, dataset, acc, tr, tr_pct, tot, t in rows:
    print(f"{mode:<12} {dataset:<8} {acc:>10.4f} {tr:>18,} {tr_pct:>11.4f}% {tot:>14,} {t:>11.1f}m")

In [ ]:
# Print raw JSON for each completed experiment so results are preserved in notebook output
import glob
for path in sorted(glob.glob("results/*.json")):
    print(f"\n{'='*60}\n{path}\n{'='*60}")
    with open(path) as f:
        print(f.read())

## 5. LoRA Weight Analysis

Peering inside the trained SST-2 LoRA model to visualise what the low-rank adaptation matrices actually learned.

### What Has the Model Learned? (ΔW Heatmap)

For every LoRA layer the effective weight update is **ΔW = B · A · (α/r)**. Plotting this matrix as a heatmap reveals which input→output connections were modified most during fine-tuning. Intense red/blue cells are connections the model changed significantly from the pretrained RoBERTa baseline; near-zero (white) cells were barely touched. We show layers 0, 5, and 11 (first, middle, last transformer block) for both the **query** and **value** projections.

In [ ]:
import sys, os
import torch

sys.path.insert(0, "code")
from models import get_lora_model
from plot_model import plot_lora_heatmaps

# ── SST-2 ─────────────────────────────────────────────────────────────────
model_sst2 = get_lora_model(num_labels=2)
model_dir  = "models/lora_sst2"

if os.path.exists(f"{model_dir}/model.safetensors"):
    from safetensors.torch import load_file
    sd = load_file(f"{model_dir}/model.safetensors")
else:
    sd = torch.load(f"{model_dir}/pytorch_model.bin", map_location="cpu")

model_sst2.load_state_dict(sd)
model_sst2.eval()

plot_lora_heatmaps(
    model=model_sst2,
    layer_indices=[0, 5, 11],
    proj_types=["query", "value"],
    title="$\\Delta W = B \\cdot A \\cdot (\\alpha/r)$  —  SST-2, RoBERTa-base, r=8",
)

#### MNLI

Repeating the same analysis for the MNLI-trained model. Because MNLI is a harder 3-way classification task across diverse genres, the learned ΔW patterns may look noticeably different from SST-2.

In [ ]:
# ── MNLI ──────────────────────────────────────────────────────────────────
model_mnli = get_lora_model(num_labels=3)
model_dir  = "models/lora_mnli"

if os.path.exists(f"{model_dir}/model.safetensors"):
    from safetensors.torch import load_file
    sd = load_file(f"{model_dir}/model.safetensors")
else:
    sd = torch.load(f"{model_dir}/pytorch_model.bin", map_location="cpu")

model_mnli.load_state_dict(sd)
model_mnli.eval()

plot_lora_heatmaps(
    model=model_mnli,
    layer_indices=[0, 5, 11],
    proj_types=["query", "value"],
    title="$\\Delta W = B \\cdot A \\cdot (\\alpha/r)$  —  MNLI, RoBERTa-base, r=8",
)

### Layer-wise Update Magnitude

The Frobenius norm of **ΔW = B · A · (α/r)** for every transformer layer shows *quantitatively* how much each layer was modified during fine-tuning. Early layers tend to stay closer to the pretrained baseline, while deeper layers specialise more for the downstream task.

In [ ]:
from plot_model import plot_layer_update_magnitude

plot_layer_update_magnitude(
    model=model_sst2,
    proj_types=["query", "value"],
    title=r"$\|\Delta W\|_F$ per Layer  —  SST-2, RoBERTa-base, r=8",
)

In [ ]:
plot_layer_update_magnitude(
    model=model_mnli,
    proj_types=["query", "value"],
    title=r"$\|\Delta W\|_F$ per Layer  —  MNLI, RoBERTa-base, r=8",
)

### Full Fine-Tune Update Magnitude

For full fine-tuning **ΔW = W_finetuned − W_pretrained** at every layer. Comparing these Frobenius norms directly against the LoRA norms above reveals how much larger the unconstrained updates are — and whether the two methods concentrate their changes at the same depths.

In [ ]:
from plot_model import plot_finetune_layer_update_magnitude

pretrained_sst2 = get_baseline_model(num_labels=2)
pretrained_sst2.eval()

finetuned_sst2 = get_baseline_model(num_labels=2)
model_dir = "models/finetune_sst2"
if os.path.exists(f"{model_dir}/model.safetensors"):
    sd = load_file(f"{model_dir}/model.safetensors")
else:
    sd = torch.load(f"{model_dir}/pytorch_model.bin", map_location="cpu")
finetuned_sst2.load_state_dict(sd)
finetuned_sst2.eval()

plot_finetune_layer_update_magnitude(
    pretrained_model=pretrained_sst2,
    finetuned_model=finetuned_sst2,
    proj_types=["query", "value"],
    title=r"$\|\Delta W\|_F$ per Layer  —  SST-2, Full Fine-Tune",
)

In [ ]:
pretrained_mnli = get_baseline_model(num_labels=3)
pretrained_mnli.eval()

finetuned_mnli = get_baseline_model(num_labels=3)
model_dir = "models/finetune_mnli"
if os.path.exists(f"{model_dir}/model.safetensors"):
    sd = load_file(f"{model_dir}/model.safetensors")
else:
    sd = torch.load(f"{model_dir}/pytorch_model.bin", map_location="cpu")
finetuned_mnli.load_state_dict(sd)
finetuned_mnli.eval()

plot_finetune_layer_update_magnitude(
    pretrained_model=pretrained_mnli,
    finetuned_model=finetuned_mnli,
    proj_types=["query", "value"],
    title=r"$\|\Delta W\|_F$ per Layer  —  MNLI, Full Fine-Tune",
)

### LoRA vs Full Fine-Tune: Side-by-Side

Overlaying both methods on the same axes makes the scale difference immediately visible. Solid lines are full fine-tuning; dashed lines are LoRA.

In [ ]:
from plot_model import plot_lora_vs_finetune_magnitude

plot_lora_vs_finetune_magnitude(
    lora_model=model_sst2,
    pretrained_model=pretrained_sst2,
    finetuned_model=finetuned_sst2,
    proj_types=["query", "value"],
    title=r"LoRA vs Full Fine-Tune: $\|\Delta W\|_F$ per Layer  —  SST-2",
)

In [ ]:
plot_lora_vs_finetune_magnitude(
    lora_model=model_mnli,
    pretrained_model=pretrained_mnli,
    finetuned_model=finetuned_mnli,
    proj_types=["query", "value"],
    title=r"LoRA vs Full Fine-Tune: $\|\Delta W\|_F$ per Layer  —  MNLI",
)

In [ ]:
# Download result JSONs to your local machine
from google.colab import files
import glob

for path in sorted(glob.glob("results/*.json")):
    files.download(path)

## 5. Rank Ablation (r = 2, 4, 8, 16)

Sweeps LoRA rank across [2, 4, 8, 16] to show the accuracy vs. trainable
parameter count tradeoff. Results saved to `results/ablation_{dataset}.json`.

In [ ]:
# ── Ablation config ─────────────────────────────────────────────────────────
ABLATION_RANKS   = [2, 4, 8, 16]   # ranks to sweep
ABLATION_DATASET = "sst2"          # "sst2" or "mnli"
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
# Run one cell per rank — each saves independently to results/ablation_r{rank}_{dataset}.json
for rank in ABLATION_RANKS:
    !python code/ablation.py \
        --rank {rank} \
        --dataset {ABLATION_DATASET} \
        --train_samples {TRAIN_SAMPLES} \
        --val_samples {VAL_SAMPLES}

In [ ]:
# Compare all saved rank results
!python code/ablation.py --compare --dataset {ABLATION_DATASET}